In [ ]:
# Importing Libraries

import numpy as np
import matplotlib.pyplot as plt

import lightkurve as lk
import nifty_ls
from astropy import units as u, constants as c
from scipy.optimize import minimize as minz
from scipy.optimize import differential_evolution as dif_evo
from scipy.optimize import Bounds
from os.path import isfile
import dill
from star_rev import neg_lnL, sum_l0_lorentzians, sum_mixed_lorentzians, pure_modes, lorentzian_single, sum_real_lorentzian, sum_complex_lorentzian

In [59]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
ex = 60
star = "KIC 7341231"
filename = f"{star}_{ex}.npy"

if(not isfile(filename)):
    search_result = lk.search_lightcurve("KIC 7341231", mission="Kepler", exptime= ex)

    lcs = search_result.download_all()
    lc = lcs.stitch().flatten().remove_outliers()

    with open(filename, "wb") as f:
        dill.dump(lc, f)
else:
    with open(filename, "rb") as f:
        lc = dill.load(f)

In [ ]:
a = np.argsort(lc.time.value)

periodogram = nifty_ls.lombscargle(t=lc.time[a].value,
                                        y=lc.flux[a].value,
                                        dy = lc.flux_err[a].value,
                                        normalization='psd',
                                        nthreads = 4,
                                        nyquist_factor=1
                                        )

In [62]:
Ny_freq = (1/(2*ex))*(10**6)

print("Nyquist frequency: " + str(Ny_freq))

ν_range = np.asarray((periodogram.freq() / u.d).to(u.uHz))


P_obs = periodogram.power[:len(ν_range)]

q_1 = 0.379
a = 0.145434
Δν = 29
ΔΠ = 111.18 / 1e6
ε_p_0 = 0.256
δν1 = -0.118
δν2 = 3.413
ε_g_1 = 0.31
ε_g_2 = 0.138
ν_max = 408
Γ_p = 1
N = 3000
ν_core = (781)/1000 # Write into parenthesis in nanohertz, converted to microhertz
ν_env = (53)/1000 # Write into parenthesis in nanohertz, converted to microhertz
inc_angle = 85
Vis_1 = 1.54
Vis_2 = 0.58

initial_params = [q_1, a, Δν, ΔΠ, ε_p_0, δν1, δν2, ε_g_1, ε_g_2, ν_max, Γ_p, N, ν_core, ν_env, inc_angle, Vis_1, Vis_2]

bounds = [
    (0.378, 0.380), # q_1
    (0.1, 0.2), # a, where q2 = a * bound given by q1
    (28.99, 29.01), # Δν
    (111.155 / 1e6, 111.205 / 1e6), # ΔΠ
    (0.252, 0.260), # ε_p_0
    (-0.138, -0.098), # δν1
    (3.392, 3.434), # δν2
    (0.304, 0.316), # ε_g_1
    (0.134, 0.142), # ε_g_2
    (393, 423), # ν_max
    (0.001, 10), # Γ_p
    (1, 6000),  # N
    (0.781, 0.781), # ν_core
    (0.053, 0.053), # ν_env
    (80, 90), # inc_angle
    (1.54, 1.54), # Vis_1
    (0.58, 0.58) # Vis_2
]

Nyquist frequency: 8333.333333333334


In [63]:
# ν_mask = (ν_range >= ν_max-4*Δν) & (ν_range <= ν_max+4*Δν)

# plt.loglog(ν_range[ν_mask], P_obs[ν_mask])
# plt.show()

In [64]:
%matplotlib tk

In [65]:
if __name__ == '__main__':
    dif_result = dif_evo(neg_lnL, bounds = bounds,  args = (P_obs, ν_range), x0 = initial_params, workers = 4, updating = "deferred")

In [66]:
free_bounds = bounds
free_bounds[13] = (0.777, 0.785)   # ν_core
free_bounds[14] = (0.047, 0.059)   # ν_env
free_bounds[15] = (1.0, 2.5)   # Vis_1
free_bounds[16] = (0.2, 1.0)   # Vis_2


if __name__ == "__main__":
    result = minz(neg_lnL, x0 = dif_result.x, args = (P_obs, ν_range), method = 'L-BFGS-B', bounds = free_bounds)

q_1, a, Δν, ΔΠ, ε_p_0, δν1, δν2, ε_g_1, ε_g_2, ν_max, Γ_p, N, ν_core, ν_env, inc_angle, Vis_1, Vis_2 = result.x

In [72]:
q_1, a, Δν, ΔΠ, ε_p_0, δν1, δν2, ε_g_1, ε_g_2, ν_max, Γ_p, N, ν_core, ν_env, inc_angle, Vis_1, Vis_2 = result.x

In [68]:
variables = [np.float64(0.37853843650061025),
            np.float64(0.11543531730102367), 
            np.float64(28.992747804783132), 
            np.float64(0.000111205), 
            np.float64(0.2528823479685966), 
            np.float64(-0.10670705114045856), 
            np.float64(3.419716153259611), 
            np.float64(0.3144085835567923), 
            np.float64(0.13815042177480547), 
            np.float64(394.5848529696586), 
            np.float64(8.309375973264178), 
            np.float64(131.15165570660156), 
            np.float64(0.781), 
            np.float64(0.7770092465244324), 
            np.float64(0.059), 
            np.float64(1.3400447450487332), 
            np.float64(0.42653771866482937)] # An example of fitted parameters. If running all cells, these will be overridden.
variables = [q_1, a, Δν, ΔΠ, ε_p_0, δν1, δν2, ε_g_1, ε_g_2, ν_max, Γ_p, N, ν_core, ν_env, inc_angle, Vis_1, Vis_2]
print(variables)

[np.float64(0.37834152260399934), np.float64(0.1806220630916454), np.float64(28.99002318566357), np.float64(0.000111205), np.float64(0.252), np.float64(-0.10248046848253897), np.float64(3.4276129401410773), np.float64(0.3116765850013555), np.float64(0.14039570810032362), np.float64(395.533907362031), np.float64(9.197418696362668), np.float64(154.7998587137042), np.float64(0.781), np.float64(0.777), np.float64(0.059), np.float64(1.3085211223843167), np.float64(0.3992330455778937)]


In [69]:
E = {
        (1,0): lambda i: np.cos(i)**2,
        (1,1): lambda i: (1/2)*np.sin(i)**2,
        (2,0): lambda i: (1/4)*(3*np.cos(i)**2-1)**2,
        (2,1): lambda i: (3/8)*np.sin(2*i)**2,
        (2,2): lambda i: (3/8)*np.sin(i)**4
    }

In [73]:
# q_1 = 0.379
# a = 0.145434
# Δν = 29
# ΔΠ = 111.18 / 1e6
# ε_p_0 = 0.256
# δν1 = -0.118
# δν2 = 3.413
# ε_g_1 = 0.31
# ε_g_2 = 0.138
# ν_max = 408
Γ_p = .1
N = 4000
# ν_core = 781 # Write in nanohertz, converted to microhertz
# ν_env = 53 # Write in nanohertz, converted to microhertz
# inc_angle = 85
# Vis_1 = 1/2
# Vis_2 = .17

fig, ax = plt.subplots(2, 1, sharex=True)

i = np.radians(inc_angle)
eps = 1e-8

ε_p_1 = ε_p_0 - 1/2 - (δν1/Δν)
ε_p_2 = ε_p_0 - 1 - (δν2/Δν)

σ_gran = 100 * (ν_max/30)**(-0.61) # Granular Amplitude
τ_gran = 3000 * (ν_max / 30)**(-0.81) # Granular Timescale
W_noise = 2.0 # Whitenoise

# May change delta nu factor for stars other than KIC 7341231
ν_mask = (ν_range >= ν_max-4*Δν) & (ν_range <= ν_max+4*Δν)
ν_sub = ν_range[ν_mask]
P_obs_sub = P_obs[ν_mask]

σ_env = 0.66*(ν_max)**(0.88)/np.sqrt(8*np.log(2)) # σ, W_env = 0.66*(ν_max)**(0.88)
gauss = np.exp((-1/2)*((ν_sub-ν_max)/(σ_env))**2)

νp_0 = pure_modes(Δν, ΔΠ, ε_p_0 + (1/2), ε_g_1, ν_max)[0]
P_l0_mod = sum_l0_lorentzians(ν_sub, νp_0, Γ_p)

νp_1, νg_1 = pure_modes(Δν, ΔΠ, ε_p_1 + (1/2), ε_g_1, ν_max)
P_l1_mod = sum(sum_mixed_lorentzians(ν_sub, q_1, Δν, ΔΠ, Γ_p, m*ν_core, m*ν_env, νp_1, νg_1)[0]*E[(1,abs(m))](i) for m in range(-1, 2)) * Vis_1
νmixed_1 = sum_mixed_lorentzians(ν_sub, q_1, Δν, ΔΠ, Γ_p, ν_core, ν_env, νp_1, νg_1)[1]

νp_2, νg_2 = pure_modes(Δν, ΔΠ/np.sqrt(3), ε_p_2 + 1, ε_g_2, ν_max)
P_l2_mod = sum(sum_mixed_lorentzians(ν_sub, q_1, Δν, ΔΠ/np.sqrt(3), Γ_p, m*ν_core, m*ν_env, νp_2, νg_2)[0]*E[(2,abs(m))](i) for m in range (-2, 3)) * Vis_2
νmixed_2 = sum_mixed_lorentzians(ν_sub, q_1, Δν, ΔΠ/np.sqrt(3), Γ_p, ν_core, ν_env, νp_2, νg_2)[1]

P_back = W_noise + (4 * σ_gran**2 * τ_gran * 1e-6) / (1 + (2 * np.pi * ν_sub * τ_gran * 1e-6)**2)

P_mod = ((P_l0_mod + P_l1_mod + P_l2_mod) * N * gauss)  + P_back

plt.xlim(275, 525)
plt.xlabel(r'Frequency $[ \mu Hz]$')
plt.ylabel(r'Power $[ppm^{2}/ \mu Hz]$')
ax[0].plot(ν_sub, P_obs_sub, lw = 1.5)
ax[1].plot(ν_sub, P_mod, c = "grey", linestyle = '--')
for v_0 in νp_0:
    ax[1].axvline(float(v_0), lw = 0.5, c = 'green')
for v_1 in νmixed_1:
    ax[1].axvline(float(v_1), lw = 0.5, c = 'blue')
for v_2 in νmixed_2:
    ax[1].axvline(float(v_2), lw = 0.5, c = 'purple')
plt.show()

-ln(L) list: (26/04/2026)

real_lorentzian (Liagre): 394524.66
real_lorentzian (Fitted): 384651.44

complex_lorentzian (Liagre): 398873.08
complex_lorentzian (Fitted): 395047.73


In [ ]:
for v in np.concatenate([νp_0,νmixed_1,νmixed_2])[ν_mask]:
    if v>500:
        print(v)

514.924
543.924
524.3445970405337
552.7456119712402
583.5199894250926
1079.7952648506557
1088.136452203498
1097.3959589198316
1111.7532069265903
1122.2918456064838
1138.7833477311312
1150.491771550978
1169.0727157230874
1182.672186638341
1203.695365346515
1226.883043550269
1243.3761059579454
1269.910774081997
1298.327577711564
1328.887155697249
521.3897776306717
549.4663397464117
578.2511440828074
1080.3319083430965
1085.7052026338724
1091.2986547042524
1096.7185161922914
1105.8696143208101
1111.8455872822394
1118.3750659940158
1124.8233613853006
1135.3109833630474
1142.3760393884345
1150.0082290131816
1162.1259688404882
1169.6459747310244
1178.3505677357016
1191.8950717849166
1200.7473491006167
1210.5056046496957
1225.4941733428439
1236.4166787163279
1252.8868410568846
1264.8380988578679
1282.937368744737
1296.1659350826326
1316.0511000257602


In [ ]:
# plt.ylim(0, 2000)
# plt.xlim(365, 405)
# plt.plot(ν_range, P_obs, lw = 1.5)
# plt.axhline(1200, c = "red", lw = 1, xmin = 0.13, xmax=0.885)
# plt.xlabel(r'Frequency $[ \mu Hz]$')
# plt.ylabel(r'Power $[ppm^{2}/ \mu Hz]$')

In [ ]:
# q_1 = 0.379
# a = 0.145434
# Δν = 29
# ΔΠ = 111.18 / 1e6
# ε_p_0 = 0.256
# δν1 = -0.118
# δν2 = 3.413
# ε_g_1 = 0.31
# ε_g_2 = 0.138
# ν_max = 408
# Γ_p = 1
# N = 600
# ν_core = 781 # Write in nanohertz, converted to microhertz
# ν_env = 53 # Write in nanohertz, converted to microhertz
# inc_angle = 85
# Vis_1 = 1/2
# Vis_2 = .17

n_init = 9
n_orders = np.arange(n_init, n_init+7)

fig, axes = plt.subplots(len(n_orders), 1, sharex=True, sharey=True, figsize=(10, 8))
plt.subplots_adjust(hspace=0)

for i, ax in enumerate(axes[::-1]):
    n_p = n_orders[i]
    
    freq_min = (n_p + ε_p_0 - 0.2) * Δν
    freq_max = (n_p + ε_p_0 + 0.8) * Δν

    m = (ν_range >= freq_min) & (ν_range <= freq_max)
    x = (ν_range[m] / Δν) - (n_p + ε_p_0)

    ν_sub = ν_range[m]
    P_obs_sub = P_obs[m]

    νp_0 = pure_modes(Δν, ΔΠ, ε_p_0 + (1/2), ε_g_1, ν_max)[0]
    P_l0_mod = sum_l0_lorentzians(ν_sub, νp_0, Γ_p)

    νp_1, νg_1 = pure_modes(Δν, ΔΠ, ε_p_1 + (1/2), ε_g_1, ν_max)
    P_l1_mod = sum(sum_mixed_lorentzians(ν_sub, q_1, Δν, ΔΠ, Γ_p, m*ν_core, m*ν_env, νp_1, νg_1)*E[(1,abs(m))](i) for m in range(-1, 2)) * Vis_1

    νp_2, νg_2 = pure_modes(Δν, ΔΠ/np.sqrt(3), ε_p_2 + 1, ε_g_2, ν_max)
    P_l2_mod = sum(sum_mixed_lorentzians(ν_sub, q_1, Δν, ΔΠ/np.sqrt(3), Γ_p, m*ν_core, m*ν_env, νp_2, νg_2)*E[(2,abs(m))](i) for m in range (-2, 3)) * Vis_2
    
    W = 0.66*(ν_max)**(0.88)
    σ = W/np.sqrt(8*np.log(2)) # σ
    gauss = np.exp((-1/2)*((ν_sub-ν_max)/(σ))**2)

    y = (P_l0_mod + P_l1_mod + P_l2_mod) * N * gauss

    ax.plot(x, P_obs_sub, color='black', lw=0.5, alpha=0.8)
    ax.plot(x, y, color='red', lw=1)
    
    # Formatting
    ax.set_ylabel(r"$n_p = $" + str(n_p), rotation=0, labelpad=20)
    ax.set_xlim(-0.2, 0.8)
    
    ax.set_yticks([])

axes[-1].set_xlabel(r"$\nu / \Delta\nu - (n_p + \epsilon_p)$, with $\epsilon_p \approx " + f"{round(ε_p_0,4)}$")
plt.show()

TypeError: can't multiply sequence by non-int of type 'numpy.float64'